# ShelfTwin: the full pipeline on Google Colab (free tier, resumable)

One notebook runs everything: **engine simulation and calibration → SKU-110K data → pack+gap detector training →
product embedder training → end-to-end pipeline demo on a real shelf photo → live dashboard**.

### Free Colab reality
One T4 (not always available), sessions last at most ~12 h and often less, idle tabs disconnect after ~90 min,
~12 GB RAM, the disk is wiped every session. So:
* **Everything is saved to Google Drive** (`MyDrive/ShelfTwin`, about 4 GB): checkpoints, prepared datasets (as single tar
  files) and results.
* **After any disconnect: Runtime → Run all.** Finished stages are skipped, prepared data is restored from Drive in seconds,
  and training resumes from the last epoch with the same schedule.
* Each session trains for at most `DETECTOR_HOURS` + `EMBED_HOURS`, then runs the demo with the best weights so far,
  so you get results every session.

### Setup (once)
1. *Runtime → Change runtime type → **T4 GPU***.
2. **Kaggle token** (needed to download SKU-110K quickly and RP2K): kaggle.com → *Settings → API → Create New Token*.
   Then in Colab, the 🔑 *Secrets* sidebar → add `KAGGLE_USERNAME` and `KAGGLE_KEY` (toggle notebook access on).
   Without a token, SKU-110K is streamed from the official archive (slower) and the embedder stage is skipped
   (the demo then uses zero-shot DINOv2).
3. Optional: already trained on Kaggle? Put `pack_detector_sku110k.pt` / `pack_gap.pt` / `embedder_best.pt` /
   `identity_thresholds.json` into `MyDrive/ShelfTwin/models/` and they are used instead of training from scratch.
4. *Runtime → Run all*. Keep the tab open.

In [ ]:
# ================= configuration =================
SMOKE_TEST = False          # True: tiny data + 1-2 epochs everywhere (~20 min) to check the whole notebook
RUN = dict(engine=True, data=True, detector=True, embedder=True, demo=True, dashboard=True)
DETECTOR_EPOCHS = 30        # total schedule; spans sessions
DETECTOR_HOURS = 3.0        # detector training per session (resumes next session)
EMBED_HOURS = 1.0           # embedder training per session (resumes next session)
EMBED_EPOCHS = 12
MAX_TRAIN_IMAGES = 4000     # SKU-110K source photos used for detector training (x2 crops each)
TEST_IMAGES = 1000          # SKU-110K test photos kept for evaluation (all 2,936 = slower eval)
SRC_SIDE, IMGSZ = 896, 640

In [ ]:
import os, sys, time, json, subprocess, shutil
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ShelfTwin"
except Exception as e:
    print("WARNING: Google Drive not mounted -> nothing persists across sessions:", e)
    BASE = "/content/ShelfTwin"
os.environ.update(ST_WORK=BASE, ST_TMP="/content/tmp", ST_INPUT_ROOT="/content/inputs")
for d in (BASE, BASE + "/models", BASE + "/outputs", "/content/tmp", "/content/inputs"):
    os.makedirs(d, exist_ok=True)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc; free -g | head -2; df -h /content | tail -1",
                     shell=True, capture_output=True, text=True).stdout)

In [ ]:
import os, sys, subprocess
def _ultra_ok():
    # importlib.metadata does not import the package, so an old version never gets cached in this kernel
    from importlib.metadata import version, PackageNotFoundError
    try:
        s = version("ultralytics")
    except PackageNotFoundError:
        return False, None
    try:
        v = tuple(int("".join(ch for ch in x if ch.isdigit()) or 0) for x in s.split(".")[:3])
    except Exception:
        return False, s
    return v >= (8, 4, 134), s
# Ultralytics >= 8.4.134 retries dense-image label assignment per image on GPU after an OOM
# instead of a very slow CPU fallback (SKU-110K has 150-700 boxes per image).
ok, ver = _ultra_ok()
print("ultralytics installed:", ver)
if not ok:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics>=8.4.134"])
    if r.returncode != 0:
        print("WARNING: could not install ultralytics>=8.4.134 (Internet on?). Trying any version.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "timm", "fastapi", "uvicorn"])

In [ ]:
# ShelfTwin code bundle (shelftwin/, sim/, notebook helpers), built by notebooks/build_notebooks.py
import base64, io, zipfile
CODE_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).extractall('/content/st')
for p in ('/content/st', '/content/st/src'):
    if p not in sys.path: sys.path.insert(0, p)
import kaggle_common as kc, colab_data as cd
kc.quiet_loggers()
WORK, TMP, OUT = kc.WORK, kc.TMP, kc.WORK / 'outputs'
PROGRESS = WORK / 'progress.json'
progress = json.loads(PROGRESS.read_text()) if PROGRESS.exists() else {}
def save_progress(**kw):
    progress.update(kw); PROGRESS.write_text(json.dumps(progress, indent=2, default=str))
print('code ready | previous progress:', progress)

## 1 · Temporal engine: simulation + calibration (CPU, parallel, resumable)

In [ ]:
if RUN["engine"] and not (OUT / "engine_calibration.json").exists():
    import glob

    import json, time
    from pathlib import Path
    from multiprocessing import Pool
    from sim.simulate import SimConfig, run, print_table
    from shelftwin.engine import EngineConfig
    import calibrate_engine as ce
    OUT = Path(os.environ.get("ST_WORK", "/kaggle/working")) / "outputs"
    OUT.mkdir(parents=True, exist_ok=True)
    sim_cfg = SimConfig(hours=1.0) if SMOKE_TEST else SimConfig()
    N_RUNS = 2 if SMOKE_TEST else 20
    t0 = time.time()
    base = run(N_RUNS, sim_cfg, verbose=False)
    print_table(base); print(f"({time.time() - t0:.0f}s)")
    grid = dict(llr_empty=[1.6], llr_stocked=[2.2], act_threshold=[3.5, 4.0], decay_per_s=[0.02]) if SMOKE_TEST else ce.GRID
    t0 = time.time()
    best, allres = ce.grid_search(sim_cfg, grid, seeds=ce.TUNE_SEEDS[:2] if SMOKE_TEST else ce.TUNE_SEEDS,
                                  cache_path=OUT / ("engine_grid_smoke.jsonl" if SMOKE_TEST else "engine_grid.jsonl"))
    print(f"{len(allres)} configs in {time.time() - t0:.0f}s on {os.cpu_count()} CPUs")
    print("best on tuning seeds:", best["params"])
    tuned = run(N_RUNS, sim_cfg, EngineConfig(**best["params"]), only=["ShelfTwin engine"], verbose=False)
    print_table({"ShelfTwin (spec defaults)": base["ShelfTwin engine"], "ShelfTwin (tuned)": tuned["ShelfTwin engine"]})
    rec = {}
    for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) + glob.glob("/content/inputs/**/*.csv", recursive=True):
        try:
            import pandas as pd
            if {"slot_id", "ts", "coverage", "visibility", "needs_restock"} <= set(pd.read_csv(p, nrows=1).columns):
                rec[p] = dict(default=ce.evaluate_recorded_csv(Path(p), EngineConfig()),
                              tuned=ce.evaluate_recorded_csv(Path(p), EngineConfig(**best["params"])))
                print(p, rec[p])
        except Exception as e:
            print("skip", p, e)
    import matplotlib.pyplot as plt
    names = list(base) + ["ShelfTwin (tuned)"]
    vals = [base[n]["false_alerts_per_cam_hour"] for n in base] + [tuned["ShelfTwin engine"]["false_alerts_per_cam_hour"]]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.barh(names, vals); ax.set_xscale("log"); ax.set_xlabel("false alerts per camera-hour (log scale)")
    for i, v in enumerate(vals): ax.text(v, i, f" {v:.2f}", va="center")
    plt.tight_layout(); plt.savefig(OUT / "engine_false_alerts.png", dpi=120); plt.show()
    json.dump(dict(baselines=base, tuned=tuned, best_params=best["params"], grid=grid, recorded=rec,
                   sim_config=sim_cfg.__dict__, n_runs=N_RUNS), open(OUT / "engine_calibration.json", "w"), indent=2, default=str)
    print("ALL DONE")
    
    save_progress(engine="done")
else:
    print("engine stage: done earlier or disabled ->", OUT / "engine_calibration.json")

## 2 · Data: SKU-110K → resized dataset → pack+gap crops (parallel, cached on Drive)

In [ ]:
import sku110k_prep as sp, gap_synth as gs
sfx = "_smoke" if SMOKE_TEST else ""
src_name = f"sku110k_{SRC_SIDE}_{MAX_TRAIN_IMAGES}_{TEST_IMAGES}{sfx}"
pg_name = f"packgap_{IMGSZ}_{MAX_TRAIN_IMAGES}{sfx}"
src_dir = pg_dir = None
if RUN["data"] or RUN["detector"] or RUN["demo"]:
    def build_src(d):
        cd.setup_kaggle_credentials()
        cd.kaggle_download(["thedatasith/sku110k-annotations"], "sku110k", min_free_gb=35)
        lim = dict(train=40, val=10, test=10) if SMOKE_TEST else dict(train=MAX_TRAIN_IMAGES, test=TEST_IMAGES)
        sp.prepare_sku110k(out=d, max_side=SRC_SIDE, limits=lim, allow_download=True)   # streams if no Kaggle copy
        cd.remove_download("sku110k")
    t0 = time.time()
    src_dir = kc.cached_dir(src_name, TMP / src_name, build_src)
    pg_dir = kc.cached_dir(pg_name, TMP / pg_name, lambda d: gs.build_pack_gap(src_dir, d, n_crops=2, out_size=IMGSZ, p_synth=0.6))
    print("data ready in", round(time.time() - t0), "s:", sp.count_split(src_dir), "| pack+gap:", sp.count_split(pg_dir))
    save_progress(data="done")

## 3 · Pack + gap detector (YOLO11s, resumable)

In [ ]:
import yolo_runner as yr
DET_NAME = "packgap_colab" + ("_smoke" if SMOKE_TEST else "")
det_weights = None
pre = [WORK / "models" / f for f in ("pack_gap.pt",)]
if any(p.exists() for p in pre) and not SMOKE_TEST:
    det_weights = next(p for p in pre if p.exists())
    print("Using your Kaggle-trained pack+gap detector:", det_weights)
elif progress.get("detector") == "done" and (OUT / "pack_gap.pt").exists() and not SMOKE_TEST:
    det_weights = OUT / "pack_gap.pt"
    print("Detector finished in an earlier session:", json.loads((OUT / "metrics_detector.json").read_text())["metrics"])
elif RUN["detector"]:
    init = WORK / "models" / "pack_detector_sku110k.pt"
    train_args = dict(
        model=str(init) if init.exists() else "yolo11s.pt", data=str(pg_dir / "data.yaml"), imgsz=IMGSZ,
        epochs=2 if SMOKE_TEST else DETECTOR_EPOCHS, batch=16, batch_per_gpu=16, name=DET_NAME,
        optimizer="AdamW", lr0=0.001 if init.exists() else 0.002, cos_lr=True, warmup_epochs=1 if SMOKE_TEST else 2,
        mosaic=0.5, close_mosaic=0 if SMOKE_TEST else 5, fliplr=0.5, degrees=0.0, hsv_v=0.5,
        max_det=1000, amp=True, patience=10, seed=0, deterministic=False, workers=os.cpu_count(),
    )
    print("initial weights:", train_args["model"])
    res = yr.train(train_args, budget_h=0.08 if SMOKE_TEST else DETECTOR_HOURS, multi_gpu=False)
    det_weights = res["best"]
    split = "test" if res["finished"] else "val"
    metrics = yr.evaluate(det_weights, pg_dir / "data.yaml", imgsz=IMGSZ, split=split, max_det=1000, iou=0.6)
    gap = gs.gap_recall_at_precision(det_weights, pg_dir, split=split, target_p=0.9, imgsz=IMGSZ,
                                     max_images=100 if SMOKE_TEST else 600)
    if res["finished"]:
        yr.export_onnx(det_weights, IMGSZ)
    kc.save_json(dict(metrics=metrics, gap_recall_at_p90=gap, finished=res["finished"], epochs_done=res["epochs_done"]),
                 OUT / "metrics_detector.json")
    shutil.copy(det_weights, OUT / "pack_gap.pt")
    save_progress(detector="done" if res["finished"] else f"{res['epochs_done']} epochs")
    print(metrics, "\ngap recall @ P0.9:", gap)
else:
    p = yr.run_dir(DET_NAME) / "weights" / "best.pt"
    det_weights = p if p.exists() else None
print("detector weights:", det_weights)

## 4 · Product embedder: DINOv2 + ArcFace on RP2K (resumable)

In [ ]:
TRAIN_HOURS = EMBED_HOURS

CFG = dict(
    seed=0, dim=256, unfreeze_blocks=4, arc_scale=64.0, arc_margin=0.5,   # 0.5 rad = 28.6 deg
    lr_backbone=1e-5, lr_head=1e-3, weight_decay=0.05,
    P=48, K=4, epochs=20, workers=None,          # workers None = all CPU cores
    max_per_class_train=40,      # caps images per training SKU to fit the free GPU quota
    max_classes=None,            # None = all SKUs in the dataset
    cache_subset=True, cache_side=256,   # one-time parallel pre-resize of the used images (much faster loading)
    multi_gpu=True,              # DataParallel over both T4s when available
    target_precision=0.98,
    time_budget_h=TRAIN_HOURS,   # per session; training resumes from the last epoch next session
    zero_shot_baseline=True, resume=True,
)
if SMOKE_TEST:
    CFG.update(epochs=2, max_classes=80, max_per_class_train=8, P=16, time_budget_h=0.1)

CFG.update(epochs=2 if SMOKE_TEST else EMBED_EPOCHS, out_dir=str(OUT), P=32)   # P=32 fits a single T4 comfortably
import embed_train as et
emb_ckpt = WORK / "models" / "embedder_best.pt"
if emb_ckpt.exists() and not SMOKE_TEST:
    print("Using your Kaggle-trained embedder:", emb_ckpt)
elif progress.get("embedder") == "done" and (OUT / "embedder_best.pt").exists() and not SMOKE_TEST:
    emb_ckpt = OUT / "embedder_best.pt"
    print("Embedder finished in an earlier session:", json.loads((OUT / "embedder_report.json").read_text()).get("test"))
elif RUN["embedder"]:
    have_cache = kc.find_cached(et.subset_key(CFG) + ".tar") is not None
    if not have_cache:
        cd.setup_kaggle_credentials()
        cd.kaggle_download(["khyeh0719/rp2k-dataset", "khyeh0719/guie-rp2k"], "rp2k", min_free_gb=20)
    try:
        report = et.train(CFG)
        emb_ckpt = OUT / "embedder_best.pt"
        if report.get("training_complete"):
            et.export_onnx(OUT)
            cd.remove_download("rp2k")
        print(json.dumps({k: report.get(k) for k in ("zero_shot_test", "test", "thresholds", "training_complete")}, indent=2))
        save_progress(embedder="done" if report.get("training_complete") else "partial")
    except FileNotFoundError as e:
        print("Embedder skipped (no RP2K; add a Kaggle token):", e)
        emb_ckpt = None
else:
    emb_ckpt = OUT / "embedder_best.pt" if (OUT / "embedder_best.pt").exists() else None
print("embedder checkpoint:", emb_ckpt or "none -> zero-shot DINOv2")

## 5 · End-to-end pipeline demo on a real shelf photo

All stages run on every frame with **your trained models**: quality gate → change gate → person mask → pack+gap
detector → geometry/coverage → identity (embedder + catalogue, abstains) → planogram alignment → temporal engine → tasks.
The scripted 8-minute episode: slot **A** runs out and is refilled, a shopper blocks **B**, glare, and products **C/D** are
swapped. The shopper is a drawn silhouette, so its mask is known (oracle with a 5 % miss rate).

In [ ]:
if RUN["demo"] and det_weights is not None:
    import cv2, numpy as np
    import e2e_demo as ed
    from shelftwin.vision.detector import PackGapDetector, DetectorConfig
    from shelftwin.vision.identity import Embedder
    det = PackGapDetector(DetectorConfig(weights=str(det_weights), imgsz=IMGSZ, device=0, half=True, conf=0.25))
    emb = Embedder(str(emb_ckpt) if emb_ckpt else None)
    th_path = next((p for p in (WORK / "models" / "identity_thresholds.json", OUT / "identity_thresholds.json") if p.exists()), None)
    th = json.loads(th_path.read_text()) if th_path else None
    # dense real test photos first
    lbl = src_dir / "labels" / "test"
    cands = sorted(lbl.glob("*.txt"), key=lambda p: -len(p.read_text().splitlines()))
    cands = [p for p in cands if 60 <= len(p.read_text().splitlines()) <= 260][:8] or cands[:8]
    demo = None
    for p in cands:
        img = cv2.imread(str(src_dir / "images" / "test" / (p.stem + ".jpg")))
        try:
            demo = ed.run_demo(ed.square_crop(img), det, embedder=emb, thresholds=th, seed=0,
                               video_path="/content/demo_raw.mp4", n_frames=60 if SMOKE_TEST else ed.N_FRAMES)
            break
        except RuntimeError as e:
            print("skip", p.stem, e)
    if demo:
        s = demo["summary"]
        print(json.dumps({k: v for k, v in s.items() if k != "tasks"}, indent=2))
        for t in s["tasks"]: print("  task:", t)
        ed.plot_trace(demo, str(OUT / "e2e_trace.png"))
        kc.save_json(s, OUT / "e2e_summary.json")
        from IPython.display import Image, HTML, display
        display(Image(str(OUT / "e2e_trace.png")))
        mp4 = OUT / "e2e_demo.mp4"
        r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/content/demo_raw.mp4", "-vcodec", "libx264",
                            "-pix_fmt", "yuv420p", "-crf", "28", str(mp4)])
        if r.returncode == 0:
            display(HTML(f'<video controls width="640" src="data:video/mp4;base64,{base64.b64encode(mp4.read_bytes()).decode()}"></video>'))
        save_progress(demo="done")
else:
    print("demo skipped (no detector weights yet or disabled)")

## 6 · Live dashboard (simulated store) inside Colab

In [ ]:
if RUN["dashboard"]:
    import threading, uvicorn, yaml
    from shelftwin.api.server import create_app
    from shelftwin.engine import EngineConfig
    from shelftwin.twin import Twin
    from sim.live import LiveStore
    cal = OUT / "engine_calibration.json"
    ecfg = EngineConfig(**json.loads(cal.read_text())["best_params"]) if cal.exists() else EngineConfig()
    live = LiveStore()
    twin = Twin(live.infos, engine_cfg=ecfg)
    app = create_app(twin, on_done=live.refill)
    def sim_loop(speed=20.0):
        while True:
            t0 = time.time()
            for sid, ts, cov, vis, q in live.step():
                twin.observe(sid, ts, cov, vis, q)
            twin.tick(live.t)
            time.sleep(max(0.0, live.dt / speed - (time.time() - t0)))
    threading.Thread(target=sim_loop, daemon=True).start()
    server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    time.sleep(3)
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(8000)   # opens the dashboard in a new tab
    except Exception as e:
        print("Open http://localhost:8000 (not running in Colab?)", e)

In [ ]:
print(json.dumps(progress, indent=2))
unfinished = [k for k in ("detector", "embedder") if RUN.get(k) and progress.get(k) not in ("done", None)]
print("ALL DONE" if not unfinished else f"Training continues next session for {unfinished}: Runtime -> Run all.")
print("Results and weights are in Google Drive:", OUT)